In [2]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("lakshmi25npathi/imdb-dataset-of-50k-movie-reviews")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'imdb-dataset-of-50k-movie-reviews' dataset.
Path to dataset files: /kaggle/input/imdb-dataset-of-50k-movie-reviews


In [3]:
import os
import numpy as np
import pandas as pd

In [4]:
files = os.listdir(path)
print(f"Files found in the directory: {files}")

# Assuming the primary dataset file is a CSV and is directly in this directory
csv_files = [f for f in files if f.endswith('.csv')]

if csv_files:
    data = os.path.join(path, csv_files[0])
    df = pd.read_csv(data)
    print(f"Successfully loaded data from {csv_files[0]} into a pandas DataFrame.")
    print("First 5 rows of the DataFrame:")
    print(df.head())
else:
    print("No CSV files found in the specified path. Please check the directory contents.")
    df = None

Files found in the directory: ['IMDB Dataset.csv']
Successfully loaded data from IMDB Dataset.csv into a pandas DataFrame.
First 5 rows of the DataFrame:
                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive


In [5]:
temp = pd.read_csv(data)

In [6]:
df = temp.iloc[:10000]

In [7]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [8]:
df['sentiment'].value_counts()

,count
sentiment,
positive,5028
negative,4972


In [9]:
df.isnull().sum()

,0
review,0
sentiment,0


In [10]:
df.duplicated().sum()

np.int64(17)

In [11]:
df.drop_duplicates(inplace = True)

/tmp/ipython-input-3424306917.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.drop_duplicates(inplace = True)


In [12]:
import re
def remove_tags(raw_text):
    cleaned_text = re.sub(re.compile('<.*?>'), '', raw_text)
    return cleaned_text

In [13]:
df['review'] = df['review'].apply(remove_tags)

/tmp/ipython-input-2336150696.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['review'] = df['review'].apply(remove_tags)


In [14]:
df['review'] = df['review'].apply(lambda x:x.lower())

/tmp/ipython-input-740760900.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['review'] = df['review'].apply(lambda x:x.lower())


In [57]:
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')
nltk.download('punkt_tab')

sw_list = stopwords.words('english')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


In [16]:
df['review'].apply(lambda x: [item for item in x.split() if item not in sw_list]).apply(lambda x : " ".join(x))

,review
0,one reviewers mentioned watching 1 oz episode ...
1,wonderful little production. filming technique...
2,thought wonderful way spend time hot summer we...
3,basically there's family little boy (jake) thi...
4,"petter mattei's ""love time money"" visually stu..."
...,...
9995,"fun, entertaining movie wwii german spy (julie..."
9996,"give break. anyone say ""good hockey movie""? kn..."
9997,movie bad movie. watching endless series bad h...
9998,"movie probably made entertain middle school, e..."


In [17]:
X = df.iloc[:,0:1]
y = df['sentiment']

In [18]:
X

,review
0,one of the other reviewers has mentioned that ...
1,a wonderful little production. the filming tec...
2,i thought this was a wonderful way to spend ti...
3,basically there's a family where a little boy ...
4,"petter mattei's ""love in the time of money"" is..."
...,...
9995,"fun, entertaining movie about wwii german spy ..."
9996,give me a break. how can anyone say that this ...
9997,this movie is a bad movie. but after watching ...
9998,this is a movie that was probably made to ente...


In [19]:
y

,sentiment
0,positive
1,positive
2,positive
3,negative
4,positive
...,...
9995,positive
9996,negative
9997,negative
9998,negative


In [20]:
from sklearn.preprocessing import LabelEncoder
encoder = LabelEncoder()
y = encoder.fit_transform(y)

In [21]:
y

array([1, 1, 1, ..., 0, 0, 1])

In [79]:
from sklearn.model_selection import train_test_split
import numpy as np

X_train, X_test, y_train, y_test = train_test_split(np.array(X), y, test_size=0.2, random_state=1)

ValueError: Found input variables with inconsistent numbers of samples: [10000, 9983]

In [23]:
X_train.shape

(7986, 1)

In [24]:
# Applying BoW
from sklearn.feature_extraction.text import CountVectorizer

In [25]:
cv = CountVectorizer()

In [26]:
X_train_bow = cv.fit_transform(X_train['review']).toarray()
X_test_bow = cv.transform(X_test['review']).toarray()

In [27]:
X_train_bow.shape

(7986, 48284)

In [29]:
from sklearn.naive_bayes import GaussianNB
gnb = GaussianNB()
gnb.fit(X_train_bow,y_train)

GaussianNB()

In [30]:
y_pred = gnb.predict(X_test_bow)
from sklearn.metrics import accuracy_score, confusion_matrix


In [31]:
accuracy_score(y_test,y_pred)

0.6364546820230346

In [34]:
confusion_matrix(y_test,y_pred)

array([[716, 236],
       [490, 555]])

Using tfidf

In [35]:
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf = TfidfVectorizer()

In [43]:
X_train_tfidf = tfidf.fit_transform(X_train['review']).toarray()
X_test_tfidf = tfidf.transform(X_test['review'])

In [46]:
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier()

In [47]:
rf.fit(X_train_tfidf,y_train)
y_pred = rf.predict(X_test_tfidf)
accuracy_score(y_test,y_pred)

0.8202303455182774

In [52]:
confusion_matrix(y_test, y_pred)


array([[787, 165],
       [194, 851]])

In [49]:
data = temp.iloc[:10000]

In [53]:
!pip install gensim
import gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 32.9 MB/s eta 0:00:00


In [54]:
from nltk import sent_tokenize
from gensim.utils import simple_preprocess

In [80]:
story = []
for doc in df['review']:
  raw = sent_tokenize(doc)
  for sent in raw:
    story.append(simple_preprocess(sent))

In [59]:
model = gensim.models.Word2Vec(
    window = 10,
    min_count = 2
)

In [60]:
model.build_vocab(story)

In [62]:
model.train(story, total_examples = model.corpus_count, epochs = model.epochs)

(8421444, 11175575)

In [63]:
len(model.wv.index_to_key)

31885

In [68]:
def doc_vector(doc):
  # Preprocess the document and remove OOV tokens
  words = [word for word in simple_preprocess(doc) if word in model.wv.index_to_key]
  if len(words) == 0:
    # Return a zero vector if all words are OOV or the document is empty
    return np.zeros(model.vector_size)
  return np.mean(model.wv[words], axis = 0)

In [69]:
doc_vector(data['review'][0])

array([-0.42817733,  0.37523353, -0.12499774,  0.06340756,  0.5215157 ,
       -0.35566372,  0.21641162,  0.6534384 , -0.3172114 , -0.03069881,
       -0.22327782, -0.4123626 , -0.2182149 ,  0.00295024, -0.20317318,
       -0.09702265, -0.37757736,  0.53502667,  0.03314309, -0.52585995,
       -0.00548571,  0.18691663,  0.0598003 , -0.46361282,  0.00434102,
        0.23332115, -0.0335182 , -0.45948792, -0.5400101 , -0.02950971,
        0.5304021 , -0.34211063, -0.4377814 , -0.31537497, -0.08608811,
        0.8284578 , -0.21522023,  0.43466914,  0.03286941, -0.34744716,
        0.18975817, -0.03006505,  0.43206415,  0.12983528,  0.36859342,
       -0.00213081, -0.08530495, -0.04472746,  0.3052361 , -0.02974391,
       -0.22058049, -0.3027491 ,  0.01142667, -0.5128738 ,  0.23023921,
        0.03096928,  0.01183397, -0.11125937, -0.24263653,  0.08491985,
        0.07337983, -0.08464477, -0.13677494, -0.00109799,  0.30818692,
        0.6303276 ,  0.09521581, -0.11415663, -0.41594884,  0.07

In [ ]:
X = []
for doc in df['review']:
  X.append(doc_vector(doc))

In [73]:
X[56]

array([-0.411441  ,  0.431463  ,  0.11266543,  0.24742806,  0.5696441 ,
       -0.43452823,  0.26603153,  0.91362995, -0.33404452, -0.09105092,
       -0.3331087 , -0.36260727, -0.20636606, -0.00817483, -0.4229869 ,
       -0.19956598, -0.5640209 ,  0.6296582 , -0.0455653 , -0.47506008,
        0.0389672 ,  0.13757196,  0.06373519, -0.43149915, -0.14702733,
        0.21979696, -0.03345745, -0.608507  , -0.72562265, -0.13003236,
        0.7014023 , -0.42826322, -0.5966098 , -0.45625275, -0.11008834,
        0.8332777 , -0.21500094,  0.5363526 ,  0.10622303, -0.16397935,
        0.3532626 , -0.00971597,  0.62458414,  0.05122606,  0.38281032,
        0.14446224, -0.04078794, -0.05708343,  0.23418497,  0.05098437,
       -0.1514121 , -0.3351244 ,  0.08226429, -0.7378992 ,  0.17278002,
        0.11661771, -0.14749864, -0.1292892 , -0.39289337, -0.06039749,
       -0.01408705,  0.07727456, -0.19788173, -0.08237675,  0.4450747 ,
        0.6588583 ,  0.03121654, -0.19152339, -0.41245174,  0.24

In [74]:
X = np.array(X)

In [75]:
X.shape

(10000, 100)

In [76]:
from sklearn.preprocessing import LabelEncoder
encoder = LabelEncoder()
y = encoder.fit_transform(df['sentiment'])

In [77]:
y

array([1, 1, 1, ..., 0, 0, 1])

In [78]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train, y_test = train_test_split(X,y, test_size =  0.2, random_state = 1)

ValueError: Found input variables with inconsistent numbers of samples: [10000, 9983]